# Loan Approval Status Classification
This notebook builds and evaluates classification models to predict loan approval status using preprocessed features. It includes model training, evaluation, ROC analysis, hyperparameter tuning, and clear outputs for reporting.

## 1. Import Required Libraries
Import all necessary libraries for modeling, evaluation, and plotting.

In [ ]:
# Import all necessary libraries for modeling, evaluation, and plotting
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn import metrics
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 2. Run Notebook 1 (Preprocessing)
Execute the preprocessing notebook to load all prepared variables (X_classification_scaled, y_classification, etc.) into this notebook's namespace.

In [ ]:
# Run Notebook 1 to execute all preprocessing steps
# This loads X_classification_scaled, y_classification, and other prepared variables
%run loan_data_preprocessing.ipynb

## 3. Load Data from Notebook 1
Verify that the preprocessed variables are available and assign them for classification.

In [ ]:
# Safety check: ensure preprocessing variables were loaded successfully
if 'X_classification_scaled' not in dir():
    raise ValueError("Preprocessed data not loaded. Run Notebook 1 first.")

# Assign preprocessed variables for classification
X = X_classification_scaled
y = y_classification

# Confirm data loaded successfully
print("Preprocessed data loaded successfully")
print("X shape:", X.shape)
print("y shape:", y.shape)

## 4. Train-Test Split
Split the data into training and testing sets with a 80/20 split, using stratification to preserve class proportions.

In [ ]:
# Split data into training (80%) and testing (20%) sets
# stratify=y ensures class proportions are preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Print shapes to verify the split
print('=' * 50)
print('TRAIN-TEST SPLIT SUMMARY')
print('=' * 50)
print(f'Training set: X_train {X_train.shape}, y_train {y_train.shape}')
print(f'Testing set:  X_test  {X_test.shape},  y_test  {y_test.shape}')
print(f'\nTraining class distribution:')
print(y_train.value_counts().sort_index())
print(f'\nTesting class distribution:')
print(y_test.value_counts().sort_index())

## 5. Define Classification Models
Create three classification models: Logistic Regression, K-Nearest Neighbors, and Gaussian Naive Bayes.

In [ ]:
# Define a dictionary of classification models to train and compare
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Gaussian NB': GaussianNB()
}

# Confirm models created
print('=== Models Defined ===')
for name, model in models.items():
    print(f'  - {name}: {model}')

## 6. Train and Evaluate Models
Train each model on the training set, make predictions on the test set, and calculate key metrics including accuracy, recall, precision, and F1-score. Print confusion matrices and classification reports.

In [ ]:
# Dictionary to store results for each model
results = {}

for name, model in models.items():
    print('=' * 60)
    print(f'MODEL: {name}')
    print('=' * 60)
    
    # --- Fit the model on training data ---
    model.fit(X_train, y_train)
    
    # --- Predict on test data ---
    y_pred = model.predict(X_test)
    
    # --- Get probability scores for ROC analysis ---
    # Use predict_proba if available, otherwise use decision_function
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)
    
    # --- Calculate evaluation metrics ---
    accuracy = metrics.accuracy_score(y_test, y_pred)
    recall = metrics.recall_score(y_test, y_pred, pos_label=1)       # pos_label=1 → Rejected loans
    precision = metrics.precision_score(y_test, y_pred, pos_label=1)
    f1 = metrics.f1_score(y_test, y_pred, pos_label=1)
    auc_score = roc_auc_score(y_test, y_score)
    
    # --- Print Confusion Matrix ---
    print('\n--- Confusion Matrix ---')
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    
    # --- Print Classification Report ---
    print('\n--- Classification Report ---')
    print(classification_report(y_test, y_pred, target_names=['Approved (0)', 'Rejected (1)']))
    
    # --- Print key metrics ---
    print('--- Key Metrics ---')
    print(f'  Accuracy:  {accuracy:.4f}')
    print(f'  Recall:    {recall:.4f}   (pos_label=1, Rejected)')
    print(f'  Precision: {precision:.4f}')
    print(f'  F1-Score:  {f1:.4f}')
    print(f'  AUC:       {auc_score:.4f}')
    
    # --- Store results for comparison later ---
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_score': y_score,
        'accuracy': accuracy,
        'recall': recall,
        'precision': precision,
        'f1': f1,
        'auc': auc_score,
        'confusion_matrix': cm
    }
    print()

## 7. ROC Curve (All Models)
Plot the ROC curves for all three models on a single chart to visually compare their classification performance. The AUC score is included in the legend.

In [ ]:
# Plot ROC curves for all models on the same chart
plt.figure(figsize=(10, 7))

for name, res in results.items():
    # Calculate the false positive rate and true positive rate for ROC
    fpr, tpr, _ = roc_curve(y_test, res['y_score'])
    auc_val = res['auc']
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc_val:.4f})")

# Plot the diagonal baseline (random classifier)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

plt.title('ROC Curves - All Models', fontsize=16, fontweight='bold')
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.legend(fontsize=11, loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Model Comparison
Compare all model metrics side-by-side in a clear tabular format.

In [ ]:
# Create a comparison table of all model metrics
print('=' * 70)
print('MODEL COMPARISON - ALL METRICS')
print('=' * 70)

# Build a DataFrame for clean tabular display
comparison_data = []
for name, res in results.items():
    comparison_data.append({
        'Model': name,
        'Accuracy': round(res['accuracy'], 4),
        'Recall': round(res['recall'], 4),
        'Precision': round(res['precision'], 4),
        'F1-Score': round(res['f1'], 4),
        'AUC': round(res['auc'], 4)
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.set_index('Model')
print(comparison_df.to_string())
print()

## 9. Select Best Model
Select the best-performing model based on **Recall**, since identifying rejected loans (positive class) is the priority in this task.

In [ ]:
# Select the best model based on RECALL (NOT accuracy)
# Recall is used because identifying rejected loans is the priority
# A missed rejected loan (false negative) is more costly than a false alarm

best_model_name = max(results, key=lambda x: results[x]['recall'])
best_model_result = results[best_model_name]

print('=' * 50)
print('BEST MODEL SELECTION')
print('=' * 50)
print(f'\nBest Model: {best_model_name}')
print(f'Recall:     {best_model_result["recall"]:.4f}')
print(f'Accuracy:   {best_model_result["accuracy"]:.4f}')
print(f'Precision:  {best_model_result["precision"]:.4f}')
print(f'F1-Score:   {best_model_result["f1"]:.4f}')
print(f'AUC:        {best_model_result["auc"]:.4f}')
print(f'\nNote: Recall is used because identifying rejected loans is the priority.')

## 10. GridSearch Hyperparameter Tuning
Tune only the best model using GridSearchCV with scoring='recall' to find the optimal hyperparameters.

In [ ]:
# Define parameter grids for each model type
param_grids = {
    'Logistic Regression': {
        'C': [0.001, 0.01, 0.1, 0.5, 1, 5, 10, 50, 100]
    },
    'KNN': {
        'n_neighbors': [3, 5, 7, 9, 11, 13, 15, 17, 19, 21]
    },
    'Gaussian NB': {
        'var_smoothing': [1e-12, 1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3]
    }
}

# Get the parameter grid for the best model
param_grid = param_grids[best_model_name]

# Create a fresh instance of the best model for tuning
if best_model_name == 'Logistic Regression':
    base_model = LogisticRegression(max_iter=1000, random_state=42)
elif best_model_name == 'KNN':
    base_model = KNeighborsClassifier()
else:
    base_model = GaussianNB()

# Run GridSearchCV with recall scoring
print('=' * 50)
print(f'GRIDSEARCH TUNING: {best_model_name}')
print('=' * 50)
print(f'Parameter grid: {param_grid}')
print(f'Scoring: recall')
print(f'Cross-validation folds: 5')

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    verbose=0
)

# Fit GridSearch on training data
grid_search.fit(X_train, y_train)

# Print results
print(f'\nBest Parameters: {grid_search.best_params_}')
print(f'Best CV Recall Score: {grid_search.best_score_:.4f}')

## 11. Retrain Best Model with Tuned Parameters
Train the best estimator from GridSearch and make predictions on the test set.

In [ ]:
# Get the best estimator from GridSearch (already fitted on full training data)
best_tuned_model = grid_search.best_estimator_

# Predict on test data using the tuned model
y_pred_tuned = best_tuned_model.predict(X_test)

# Get probability scores for ROC analysis
if hasattr(best_tuned_model, 'predict_proba'):
    y_score_tuned = best_tuned_model.predict_proba(X_test)[:, 1]
else:
    y_score_tuned = best_tuned_model.decision_function(X_test)

print('=' * 50)
print('TUNED MODEL RETRAINED SUCCESSFULLY')
print('=' * 50)
print(f'Model: {best_model_name}')
print(f'Best Parameters: {grid_search.best_params_}')

## 12. Evaluate Tuned Model
Calculate and display all evaluation metrics for the tuned model.

In [ ]:
# Calculate metrics for the tuned model
accuracy_tuned = metrics.accuracy_score(y_test, y_pred_tuned)
recall_tuned = metrics.recall_score(y_test, y_pred_tuned, pos_label=1)
precision_tuned = metrics.precision_score(y_test, y_pred_tuned, pos_label=1)
f1_tuned = metrics.f1_score(y_test, y_pred_tuned, pos_label=1)
auc_tuned = roc_auc_score(y_test, y_score_tuned)

print('=' * 50)
print(f'TUNED MODEL EVALUATION: {best_model_name}')
print('=' * 50)

# Print confusion matrix
print('\n--- Confusion Matrix (Tuned) ---')
cm_tuned = confusion_matrix(y_test, y_pred_tuned)
print(cm_tuned)

# Print classification report
print('\n--- Classification Report (Tuned) ---')
print(classification_report(y_test, y_pred_tuned, target_names=['Approved (0)', 'Rejected (1)']))

# Print key metrics
print('--- Key Metrics (Tuned) ---')
print(f'  Accuracy:  {accuracy_tuned:.4f}')
print(f'  Recall:    {recall_tuned:.4f}   (pos_label=1, Rejected)')
print(f'  Precision: {precision_tuned:.4f}')
print(f'  F1-Score:  {f1_tuned:.4f}')
print(f'  AUC:       {auc_tuned:.4f}')

## 13. Before vs After Comparison
Compare the model performance before and after hyperparameter tuning to see the impact of GridSearch.

In [ ]:
# Compare before and after tuning
print('=' * 60)
print(f'BEFORE vs AFTER TUNING: {best_model_name}')
print('=' * 60)

# Confusion matrix comparison
print('\n--- Confusion Matrix BEFORE Tuning ---')
print(best_model_result['confusion_matrix'])

print('\n--- Confusion Matrix AFTER Tuning ---')
print(cm_tuned)

# Metrics comparison table
print('\n--- Metrics Comparison ---')
before_after = pd.DataFrame({
    'Metric': ['Accuracy', 'Recall', 'Precision', 'F1-Score', 'AUC'],
    'Before Tuning': [
        round(best_model_result['accuracy'], 4),
        round(best_model_result['recall'], 4),
        round(best_model_result['precision'], 4),
        round(best_model_result['f1'], 4),
        round(best_model_result['auc'], 4)
    ],
    'After Tuning': [
        round(accuracy_tuned, 4),
        round(recall_tuned, 4),
        round(precision_tuned, 4),
        round(f1_tuned, 4),
        round(auc_tuned, 4)
    ]
})
before_after['Change'] = before_after['After Tuning'] - before_after['Before Tuning']
before_after = before_after.set_index('Metric')
print(before_after.to_string())

# Highlight recall comparison
print(f'\n=== RECALL COMPARISON ===')
print(f'  Before Tuning: {best_model_result["recall"]:.4f}')
print(f'  After Tuning:  {recall_tuned:.4f}')
recall_change = recall_tuned - best_model_result['recall']
if recall_change > 0:
    print(f'  Improvement:   +{recall_change:.4f}')
elif recall_change < 0:
    print(f'  Change:        {recall_change:.4f}')
else:
    print(f'  No change in recall.')

## 14. Final ROC Curve (Tuned Model)
Plot the ROC curve for the final tuned model with its AUC score.

In [ ]:
# Plot ROC curve for the final tuned model
fpr_tuned, tpr_tuned, _ = roc_curve(y_test, y_score_tuned)

plt.figure(figsize=(10, 7))
plt.plot(fpr_tuned, tpr_tuned, linewidth=2, color='#2196F3',
         label=f'{best_model_name} - Tuned (AUC = {auc_tuned:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

plt.title(f'ROC Curve - Final Tuned Model ({best_model_name})', fontsize=16, fontweight='bold')
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.legend(fontsize=12, loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Print final AUC score
print(f'Final AUC Score: {auc_tuned:.4f}')

## 15. Final Summary
Clean summary of the entire classification workflow for coursework reporting.

In [ ]:
# ========================================
# FINAL CLASSIFICATION SUMMARY
# ========================================
print('=' * 60)
print('CLASSIFICATION WORKFLOW - FINAL SUMMARY')
print('=' * 60)

print(f'\n--- Dataset ---')
print(f'  Features (X): {X.shape}')
print(f'  Target (y):   {y.shape}')
print(f'  Train size:   {X_train.shape[0]} samples')
print(f'  Test size:    {X_test.shape[0]} samples')

print(f'\n--- Models Evaluated ---')
for name in results:
    print(f'  - {name}: Recall = {results[name]["recall"]:.4f}, Accuracy = {results[name]["accuracy"]:.4f}')

print(f'\n--- Best Model ---')
print(f'  Model:           {best_model_name}')
print(f'  Selection Basis: Recall (identifying rejected loans is the priority)')

print(f'\n--- Tuning Results ---')
print(f'  Best Parameters: {grid_search.best_params_}')
print(f'  Best CV Recall:  {grid_search.best_score_:.4f}')

print(f'\n--- Final Tuned Model Performance ---')
print(f'  Accuracy:  {accuracy_tuned:.4f}')
print(f'  Recall:    {recall_tuned:.4f}   (pos_label=1, Rejected)')
print(f'  Precision: {precision_tuned:.4f}')
print(f'  F1-Score:  {f1_tuned:.4f}')
print(f'  AUC:       {auc_tuned:.4f}')

print(f'\n' + '=' * 60)
print('CLASSIFICATION COMPLETE')
print('=' * 60)